In [ ]:
import numpy as np
import os
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv1D, MaxPooling1D, Flatten, LSTM, Dense
from itertools import product
from sklearn import metrics

PATH = os.path.join('../datasets/sl2t_data/data_pranjalSir/data')
# PATH = os.path.join('../datasets/sl2t_data/data/data')
actions = np.array(os.listdir(PATH))
sequences, frames = 30, 20
label_map = {label: num for num, label in enumerate(actions)}

landmarks, labels = [], []
for action, sequence in product(actions, range(sequences)):
    temp = []
    print(f"Action: {action}")
    for frame in range(frames):
        npy = np.load(os.path.join(PATH, action, str(sequence), str(frame) + '.npy'))
        print(f"Frame: {frame}")
        print(f"Npy: {npy}")
        temp.append(npy)
    landmarks.append(temp)
    labels.append(label_map[action])

X, Y = np.array(landmarks), to_categorical(labels).astype(int)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.10, random_state=34)

from keras.layers import Input

model = Sequential([
    Input(shape=(frames, 126)),  
    Conv1D(64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(32, return_sequences=True, activation='relu'),
    LSTM(64, activation='relu'),
    Dense(actions.shape[0], activation='softmax')
])

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
history = model.fit(X_train, Y_train, epochs=100, validation_data=(X_test, Y_test))
model.save('Final.h5')

predictions = np.argmax(model.predict(X_test), axis=1)
test_labels = np.argmax(Y_test, axis=1)
accuracy = metrics.accuracy_score(test_labels, predictions)
print(f"Test Accuracy: {accuracy}")

Action: a
Frame: 0
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Frame: 1
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Frame: 2
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Frame: 3
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Frame: 4
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
Frame: 5
Npy: [[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 

FileNotFoundError: [Errno 2] No such file or directory: '../datasets/sl2t_data/data\\a\\9.npy'

In [7]:
npy

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0.])

In [1]:
import cv2
import mediapipe as mp
import numpy as np


def image_process(image, holistic):
    """
    Processes an image frame using MediaPipe Holistic
    and returns detection results.
    
    Args:
        image: Input frame from camera
        holistic: Initialized MediaPipe Holistic model
        
    Returns:
        MediaPipe Holistic results object
    """
    # Convert BGR to RGB and process with MediaPipe
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = holistic.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return results

def keypoint_extraction(results):
    """
    Extracts and concatenates keypoints from Holistic results
    into a flattened numpy array.
    
    Args:
        results: MediaPipe Holistic results object
        
    Returns:
        Flattened numpy array of keypoint coordinates (x,y,z)
    """
    # Initialize empty arrays for each component
    face = np.array([[res.x, res.y, res.z] for res in 
                    results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    
    pose = np.array([[res.x, res.y, res.z] for res in 
                    results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3)
    
    lh = np.array([[res.x, res.y, res.z] for res in 
                  results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    
    rh = np.array([[res.x, res.y, res.z] for res in 
                  results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    
    return np.concatenate([pose, face, lh, rh])

def draw_landmarks(image, results):
    """Visualizes landmarks on the image"""
    mp.solutions.drawing_utils.draw_landmarks(
        image, results.face_landmarks, mp.solutions.holistic.FACEMESH_CONTOURS)
    mp.solutions.drawing_utils.draw_landmarks(
        image, results.pose_landmarks, mp.solutions.holistic.POSE_CONNECTIONS)
    mp.solutions.drawing_utils.draw_landmarks(
        image, results.left_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)
    mp.solutions.drawing_utils.draw_landmarks(
        image, results.right_hand_landmarks, mp.solutions.holistic.HAND_CONNECTIONS)



In [ ]:
#INCLUDE ISL SignLanguage -to- Speech Conversion Real Time GUI
import numpy as np
import os
import cv2
import mediapipe as mp
from keras.models import load_model
# from KeypointsExtraction import draw_landmarks, image_process, keypoint_extraction
import keyboard
import pyttsx3  # <-- Added

# Initialize Text-to-Speech Engine
engine = pyttsx3.init()
engine.setProperty('rate', 150)  # Speech rate (optional)
engine.setProperty('volume', 1.0)  # Max volume

# Path to data and actions defined during training
PATH = os.path.join('../datasets/sl2t_data/data/data')
actions = np.array(os.listdir(PATH))

# Load the trained model
model = load_model('../app/models/Final.h5')

# Initialize prediction and sentence-related lists
sentence, keypoints, last_prediction = [], [], None
cooldown_frames, cooldown_threshold = 0, 20  # Cooldown period of 20 frames after each prediction
skip_frames_after_hand_detected, skip_counter = 5, 0  # Skip 5 frames after hand is detected

# Open camera for capturing
cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Cannot access camera.")
    exit()

with mp.solutions.holistic.Holistic(min_detection_confidence=0.70, min_tracking_confidence=0.70) as holistic:
    hand_present = False  # Track if a hand is present

    while cap.isOpened():
        # Capture frame from camera
        ret, image = cap.read()
        if not ret:
            break

        # Process frame and extract keypoints
        results = image_process(image, holistic)
        draw_landmarks(image, results)

        # Check if a hand is present in the frame
        hand_detected = results.left_hand_landmarks or results.right_hand_landmarks

        if hand_detected:
            if not hand_present:
                # Hand just appeared, start skip counter
                hand_present = True
                skip_counter = skip_frames_after_hand_detected
            elif skip_counter > 0:
                skip_counter -= 1
                continue

            # Extract keypoints after hand has been stable for 5 frames
            keypoints.append(keypoint_extraction(results))

            # Predict every 20 frames if cooldown is not active
            if len(keypoints) == 20 and cooldown_frames == 0:
                keypoints = np.array(keypoints)
                prediction = model.predict(keypoints[np.newaxis, :, :])
                keypoints = []

                if np.max(prediction) >= 0.85:
                    predicted_action = actions[np.argmax(prediction)]

                    if predicted_action != last_prediction:
                        sentence.append(predicted_action)
                        last_prediction = predicted_action
                        cooldown_frames = cooldown_threshold

                        # 🔊 Speak the predicted phrase
                        engine.say(predicted_action)
                        engine.runAndWait()

        else:
            hand_present = False
            keypoints = []

        cooldown_frames = max(0, cooldown_frames - 1)

        if len(sentence) > 7:
            sentence = sentence[-7:]

        if keyboard.is_pressed(' '):
            sentence, keypoints, last_prediction = [], [], None

        if sentence:
            sentence[0] = sentence[0].capitalize()

        # Display sentence
        display_text = ' '.join(sentence)
        text_size = cv2.getTextSize(display_text, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)[0]
        text_x = (image.shape[1] - text_size[0]) // 2
        cv2.putText(image, display_text, (text_x, 470),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

        cv2.imshow('Real-time Sign Prediction', image)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Release everything
cap.release()
cv2.destroyAllWindows()

ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "conv1d" is incompatible with the layer: expected axis -1 of input shape to have value 126, but received input with shape (1, 20, 1629)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 20, 1629), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

: 